# T4 Replication Lab: Card & Krueger (1994) Minimum Wage and Employment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/Appendix/T4_Replication_Card_Krueger_1994.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=Appendix/T4_Replication_Card_Krueger_1994.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)
## The Lens: A Policy Change as a Natural Experiment
New Jersey raised its minimum wage while neighboring eastern Pennsylvania did not. The empirical question is whether fast-food employment changed differently in New Jersey after the policy. The two-wave restaurant panel makes the canonical difference-in-differences (DiD) estimand transparent: compare the before/after change in the treated state with the same change in the control state.

**Economic question.** In *T4 Replication Lab: Card & Krueger (1994) Minimum Wage and Employment*, what must remain economically invariant when the computational representation changes? This mathematical result earns its place in the curriculum because later economic arguments rely on its hypotheses, not only its conclusion. As you read each derivation, track which assumption licenses each step and construct a counterexample when an assumption is removed. The objective is to make the theorem usable as a diagnostic tool in optimization, probability, econometrics, or dynamic models rather than a formula to memorize.

### Learning Objectives
- Reconstruct the two-by-two DiD estimand from group means.
- Estimate the same interaction coefficient by OLS with heteroskedasticity-robust uncertainty.
- Add chain/ownership/region controls without changing the target estimand.
- State what a two-wave design can and cannot reveal about parallel trends.

### Prerequisites
- `../06-Econometrics/08_Difference_in_Differences.ipynb`
- `../06-Econometrics/01_Linear_Model_and_OLS.ipynb`

### Table of Contents
1. Data provenance and validation
2. Two-by-two DiD
3. Regression representation
4. Controlled specification and diagnostics
5. Interpretation and identification limits
6. Exercises
7. Summary and references
* **Learning-path prerequisite:** [`T3_Autograding_with_Otter.ipynb`](T3_Autograding_with_Otter.ipynb)


> **Learning path:** Building on [`T3_Autograding_with_Otter.ipynb`](T3_Autograding_with_Otter.ipynb); next continue with [`T5_Replication_Fama_French_Five_Factor.ipynb`](T5_Replication_Fama_French_Five_Factor.ipynb).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def locate(relative: str) -> Path:
    candidates = [Path(relative), Path("..") / relative]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(f"Could not locate {relative} from repository root or Appendix/")
    return path

DATA = locate("data/replications/card_krueger_1994_njmin3.csv")
df = pd.read_csv(DATA)
required = {"nj", "d", "d_nj", "fte", "bk", "kfc", "roys", "wendys", "co_owned", "centralj", "southj", "pa1", "pa2"}
missing = required.difference(df.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"
assert set(df["nj"].dropna().unique()) <= {0, 1}
assert set(df["d"].dropna().unique()) <= {0, 1}
print(f"Loaded {len(df):,} restaurant-wave observations from {DATA}.")


## 1. Data Provenance and Estimand
The bundled file is a long-form version of the Card–Krueger fast-food survey: `nj=1` identifies New Jersey, `d=1` identifies the post-policy wave, `d_nj = nj × d`, and `fte` is full-time-equivalent employment. The DiD estimand is

$$
\widehat{\tau}_{DiD} = (\bar Y_{NJ,post}-\bar Y_{NJ,pre})-(\bar Y_{PA,post}-\bar Y_{PA,pre}).
$$

With only one pre-treatment wave, this dataset cannot empirically validate a pre-trend. Parallel trends remains an identifying assumption that must be defended institutionally and with external evidence.


In [ ]:
means = df.groupby(["nj", "d"], observed=True)["fte"].agg(["count", "mean"])
display(means)
pa_pre = means.loc[(0, 0), "mean"]
pa_post = means.loc[(0, 1), "mean"]
nj_pre = means.loc[(1, 0), "mean"]
nj_post = means.loc[(1, 1), "mean"]
did = (nj_post - nj_pre) - (pa_post - pa_pre)
print(f"Two-by-two DiD estimate: {did:.4f} FTE workers per restaurant")

plot = means["mean"].unstack("d").rename(columns={0: "Pre", 1: "Post"})
ax = plot.T.plot(marker="o", figsize=(8, 5))
ax.set(title="Mean FTE Employment by State and Survey Wave", xlabel="Survey wave", ylabel="FTE employment")
ax.legend(["Pennsylvania", "New Jersey"], title="Group")
plt.show()


## 2. Regression Representation
The saturated two-by-two regression is

$$
Y_{it}=\alpha+\gamma NJ_i+\lambda Post_t+\tau(NJ_i\times Post_t)+\varepsilon_{it}.
$$

In this design, the interaction coefficient $\tau$ is algebraically identical to the four-cell DiD above. Robust standard errors address heteroskedasticity, but they do not repair violations of parallel trends or other identification assumptions.


In [ ]:
base = smf.ols("fte ~ nj + d + d_nj", data=df).fit(cov_type="HC1")
print(base.summary().tables[1])
assert np.isclose(base.params["d_nj"], did, atol=1e-10)
print(f"Interaction equals manual DiD: {base.params['d_nj']:.4f}")


## 3. Controlled Specification and Diagnostics
Controls can absorb residual composition differences across restaurant chains, ownership, and regions. They should not be chosen because they make the treatment coefficient more attractive. The estimand remains the post-policy differential change associated with New Jersey.


In [ ]:
formula = "fte ~ nj + d + d_nj + bk + kfc + roys + co_owned + centralj + southj + pa1 + pa2"
controlled = smf.ols(formula, data=df).fit(cov_type="HC1")
comparison = pd.DataFrame({
    "estimate": [base.params["d_nj"], controlled.params["d_nj"]],
    "robust_se": [base.bse["d_nj"], controlled.bse["d_nj"]],
}, index=["Two-by-two", "With controls"])
display(comparison)
print(f"Complete-case N in controlled model: {int(controlled.nobs)}")


## Key Equations

- **Two-by-two DiD:** $\widehat{ATT}=(\bar Y_{NJ,post}-\bar Y_{NJ,pre})-(\bar Y_{PA,post}-\bar Y_{PA,pre})$.
- **Regression form:** $Y_{it}=\alpha+\beta NJ_i+\gamma Post_t+\delta(NJ_i\times Post_t)+u_{it}$, where $\delta$ equals the two-by-two DiD in the saturated group-time design.
- **Identification:** the causal reading of $\delta$ requires a credible parallel-trends counterfactual; two waves alone cannot test pre-treatment trend equality.


## 4. Interpretation and Identification Limits
A positive DiD coefficient means New Jersey employment rose relative to the Pennsylvania comparison group over the two survey waves. That is a statement about the observed design, not a universal claim that minimum-wage increases always raise employment. With one pre-period, the notebook cannot test differential pre-trends; survey measurement, spillovers, compositional changes, and treatment anticipation remain substantive concerns.

## Exercises
**1. Design logic (Conceptual):** Derive why the OLS interaction coefficient equals the four-cell DiD. Which assumption gives the coefficient a causal interpretation?

**2. Robustness (Applied):** Re-estimate the coefficient using alternative FTE constructions if the source variables are available, and compare robust uncertainty and sample size.

**3. Identification stress test (Challenge):** Simulate a differential pre-trend that continues into the post period. Show how the DiD coefficient mixes the policy effect with the trend violation.

## Summary & Key Takeaways
- The manual four-cell calculation and OLS interaction target the same DiD estimand.
- Robust standard errors address variance estimation, not identification.
- A two-wave design makes institutional reasoning about parallel trends especially important.

## References & Further Reading
- Card, D. & Krueger, A. B. (1994). Minimum Wages and Employment: A Case Study of the Fast-Food Industry in New Jersey and Pennsylvania. *American Economic Review*, 84(4), 772–793.
- Angrist, J. D. & Pischke, J.-S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.
